In [1]:
def get_stats(ids):
    # The goal is
    counts = {}
    for chunk in ids: # chunk ? 
        for pair in zip(chunk, chunk[1:]): 
            counts[pair] = counts.get(pair,0) +1 

    return counts
            # whNat we want if k exist 


In [2]:
def merge(ids, pair, idx):

    new_ids = [] 

    for chunk in ids:
        
        i = 0

        new_chunk_ids = []

        while i < len(chunk): 
            if len(chunk) - i > 1 and pair[0] == chunk[i] and pair[1] == chunk[i+1]:
                new_chunk_ids.append(idx)
                i += 2 

            else :
                new_chunk_ids.append(chunk[i]) 

                i += 1  

        new_ids.append(new_chunk_ids)

    return new_ids


--- 1. What finditer() gives you (Match Objects) ---
<regex.Match object; span=(0, 1), match='I'>
<regex.Match object; span=(1, 4), match="'ll">
<regex.Match object; span=(4, 8), match=' pay'>
<regex.Match object; span=(8, 10), match=' $'>
<regex.Match object; span=(10, 13), match='100'>


In [3]:
pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

compiled = re.compile(pattern)


NameError: name 're' is not defined

In [373]:
pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

compiled = re.compile(pattern)

def pretokenize(data: str, special_token : list) -> list: 

    if special_token:

        tokens = sorted(special_token, key=len, reverse=True)
        toks =  "|".join(re.escape(t) for t in tokens)
        chunks = re.split(toks, data)

    else :
        chunks = [data]

    pre_tokens = []
    for chunk in chunks:
        #compiled is where the regex pattern is 
        pre_tokens.extend(m.group(0) for m in compiled.finditer(chunk))
        

    return pre_tokens


In [374]:
import regex as re

def train(text, vocab_size,special_tokens):

    special_tokens = special_tokens or []

    length_special_token = len(special_tokens)
    
    with open (file=text, encoding="utf-8") as f:
        data = f.read()

    pre_tokens = pretokenize(data, special_tokens)

    new_token_amount = vocab_size - 256 # new token amount
    
    idx = 255 

    ids = [] 


    for pre_token in pre_tokens:
        ids.append(pre_token.encode("utf-8"))
    
    vocab = {}

    merges = []

    for i in range(256):
        vocab[i] = bytes([i])

    for _ in range(new_token_amount - len(special_tokens)):  # main training
        stats = get_stats(ids)
        if not stats:
            break 
        top_pair = max(stats, key=stats.get) # return a tuple
        idx += 1 
        ids = merge(ids,top_pair,idx)
        vocab[idx] = vocab[top_pair[0]] + vocab[top_pair[1]]
        merges.append((vocab[top_pair[0]], vocab[top_pair[1]]))

    for i in range(len(special_tokens)):

        special_token= str(special_tokens[i])
        vocab[1 + i + idx] = special_token.encode("utf-8")
        
    
    # last
    
    return vocab, merges 
    


In [375]:
data = "../data/small.txt"
stst = ["<|endoftext|>"]
vocab_size = 300
vocab, merge = train(data, vocab_size, stst)
print(vocab)


In [380]:
def encode(text:str) -> list:
    token_ids = []
 
    special_pattern ="|".join(re.escape(t) for t in special_tokens)

    pieces = re.split(f"({special_pattern})",text )

    piece_bytes = []

    for i in range(len(pieces)):
        
        piece_bytes.append(pieces[i].encode("utf-8"))

    for i in range(len(piece_bytes)):
        if piece_bytes[i] in list_specialtoken_enc:
            token_ids.append(rev_vocab[piece_bytes[i]] )
        else:  
            # token_ids.append(pieces[i])
            chunk = pieces[i] 

            pre_tokens = pretokenize(chunk,False)

            for chunk in pre_tokens:
                chunk_ids = [bytes([b]) for b in chunk.encode("utf-8")]

                while len(chunk_ids) > 1:
                    best_pair = None
                    best_pos = float("inf")

                    for ch1, ch2 in zip(chunk_ids, chunk_ids[1:]):
                        if (ch1, ch2) in merge:
                            pos = merge.index((ch1, ch2))
                            if pos < best_pos:
                                best_pos = pos
                                best_pair = (ch1, ch2)

                    if best_pair is None:
                        break

                    new_chunk_ids = []
                    j = 0
                    while j < len(chunk_ids):
                        if j < len(chunk_ids) - 1 and (chunk_ids[j], chunk_ids[j + 1]) == best_pair:
                            new_chunk_ids.append(chunk_ids[j] + chunk_ids[j + 1])
                            j += 2
                        else:
                            new_chunk_ids.append(chunk_ids[j])
                            j += 1
                    chunk_ids = new_chunk_ids

                for part in chunk_ids:
                    token_ids.append(rev_vocab[part])

            
        
    return token_ids 
    


In [381]:
def decode(ids):

    c = b''.join(vocab[token_id] for token_id in ids)
    text = c.decode("utf-8", errors="replace")
    return text


In [384]:
decode(encode(' the'))


In [ ]:
t = "hi<|endoftext|>yo<|pad|>ok<|endoftext|>"
a = encode(t)
print(a)


In [ ]:


text = "They was the<|endoftext|> and"
special_tokens = ["<|endoftext|>", "<|pad|>"]
print(t)

special_pattern ="|".join(re.escape(t) for t in special_tokens)

pieces = re.split(f"({special_pattern})",text )
print(pieces)

rev_vocab = {v:k for k,v in vocab.items() }

piece_bytes = []

for i in range(len(pieces)):
    
    piece_bytes.append(pieces[i].encode("utf-8"))

print(piece_bytes)


In [ ]:

token_ids = []
pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
list_specialtoken_enc = []
re_chunk = []


for i in range(len(stst)):
    list_specialtoken_enc.append(stst[i].encode("utf-8")) 


for i in range(len(piece_bytes)):
    if piece_bytes[i] in list_specialtoken_enc:
        token_ids.append(rev_vocab[piece_bytes[i]] )
    else:  
        # token_ids.append(pieces[i])
        chunk = pieces[i] 

        pre_tokens = pretokenize(chunk,False)
        token_ids.extend(pre_tokens)  

        for chunk in pre_tokens:
            chunk_ids = chunk.encode("utf-8")
            for ch1, ch2 in zip(chunk_ids, chunk_ids[1:]):
                
                 
#print(token_ids)
 


In [ ]:
print(merge)


In [ ]:
decode([84, 257, 121, 292, 261, 299, 265])


In [ ]:
def Transgender():
    pass    


In [ ]:


import regex as re

text = "I'll <|endoftext|>  pay $100 <|pad|> for 2 coffees!  "

# special_token = "<|endoftext|>"
special_tokens = ["<|endoftext|>", "<|pad|>"]
special_tokens_re = []

for i in range(len(special_tokens)):
    special_tokens_re.append(re.escape(special_tokens[i]))

all_special_tokens = "|".join(special_tokens_re)
chunks = re.split(all_special_tokens, text)

print(chunks)


pre_token = []
for chunk in chunks:
    pre_token.extend([m.group(0) for m in compiled.finditer(chunk)]) 
    

print(pre_token)
